# 01 影像组学对照
在保存完整影像的电脑上提取特征并训练，完成后把 `results/radiomics` 复制到算力机的同名位置，06 就能读取。

这本使用 MIRP 提取强度和纹理特征，用 scikit-image 计算二维形状。只需要 `cohort.csv`、配准三序列和人工标注，特征提取和模型训练都使用 CPU。

Python 3.12 可以使用 MIRP 2.5.0。首次使用时，在当前 Notebook 新建一个单元运行下面的安装命令，安装完成后重启内核，再从上往下运行本 Notebook：

```python
%pip install "mirp==2.5.0"
```

MIRP 会安装它需要的 scikit-image 等依赖；本 Notebook 不再导入 PyRadiomics 或 SimpleITK。选用 2.5.0，是因为它支持 Python 3.12，且其依赖要求不强制升级到 NumPy 2.4 和 pandas 3。代码不检查版本号。

库的[安装要求](https://github.com/oncoray/mirp/blob/v2.5.0/pyproject.toml)和[特征提取接口](https://oncoray.github.io/mirp/quantitative_image_analysis.html)可用于核对环境。


## 读取路径和患者清单


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("分类输入应为每例4层、3通道、224×224。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
cases = read_cases()
OUTPUT = RESULTS / "radiomics"
EXTRACTION = {"normalize": True, "normalizeScale": 100, "binWidth": 25}
RUN_EXTRACTION = False
RUN_TRAINING = False

display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))


## 读取选定四层
只读取各患者标注过的四层，按标签 1、2、3 分别提取二维特征。这里不把四层当成完整三维病灶。

当前按原始像素网格计算二维特征，形状量使用像素单位，不解释为 mm² 或 mm³。

`patients_root` 填直接包含各患者文件夹的目录；每例读取 `患者文件夹/配准图像`，患者文件夹名以 MR 编号结尾，例如 `姓名 MR250418095`。

取层仍以 MoCA 表的“脑室体素最大层”为起点。范围外有标注会显示提示，计算继续使用选定四层；用于影像组学和分割前，需要确认这四层均已完成标注，不能把未标注层当作没有病灶。


In [ ]:
import re
import nibabel as nib


def image_directories():
    root = path_from_config("patients_root")
    if not root.is_dir():
        raise FileNotFoundError(f"患者总目录不存在：{root}")

    folders = {}
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir():
            continue
        match = re.search(r"MR\s*(\d+)$", patient_dir.name.strip(), re.IGNORECASE)
        if not match:
            continue
        directory = patient_dir / "配准图像"
        if not (directory / "t1_flair.nii.gz").is_file():
            continue

        mr = "MR" + match.group(1)
        directory = directory.resolve()
        if mr in folders and folders[mr] != directory:
            raise ValueError(
                f"两个患者文件夹使用了同一个影像号 {mr}：\n"
                f"{folders[mr]}\n{directory}"
            )
        folders[mr] = directory

    if not folders:
        raise FileNotFoundError(
            f"未在 {root} 找到 患者文件夹/配准图像/t1_flair.nii.gz。"
            "请把 patients_root 设为直接包含各患者文件夹的目录。"
        )
    return folders


def read_four_slices(patient, folders=None):
    recorded = [Path(str(patient.get(key, ""))) for key in SEQUENCES + ["manual_mask"]]
    if not all(file.is_file() for file in recorded):
        folders = folders if folders is not None else image_directories()
        if patient.mr not in folders:
            raise FileNotFoundError(f"找不到{patient.mr}的三序列目录。")
        recorded = [folders[patient.mr] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    volumes = [nib.load(str(file)) for file in recorded]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, atol=tolerance, rtol=0) for v in volumes[1:]):
        raise ValueError(f"{patient.mr}三序列与标注的shape/affine不一致。")
    layer = float(patient.max_layer)
    if not np.isfinite(layer) or not layer.is_integer():
        raise ValueError(f"{patient.mr}起始层不是整数。")
    start = int(layer) - 1
    if start < 0 or start + 4 > ref.shape[2]:
        raise ValueError(f"{patient.mr}四层取层越界。")
    labels = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(labels).all() or not set(np.unique(labels)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr}标签必须为0/1/2/3。")
    if np.any(labels[:, :, :start]) or np.any(labels[:, :, start + 4:]):
        print(f"提示：{patient.mr}四层之外仍有标注；本次按Excel取第{start + 1}～{start + 4}层，其余层不参与计算。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, start:start + 4] for v in volumes[:3]]
    if any(not np.isfinite(image).all() for image in images):
        raise ValueError(f"{patient.mr}影像包含无效数值。")
    return images, labels[:, :, start:start + 4].astype(np.uint8)


def normalize_four(volume):
    lo, hi = np.percentile(volume, [1, 99.5])
    return np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0, 1)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss


def choose_threshold(labels, probabilities):
    fpr, tpr, thresholds = roc_curve(labels, probabilities)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def classification_metrics(labels, probabilities, threshold):
    labels, probabilities = np.asarray(labels, int), np.asarray(probabilities, float)
    if not np.isfinite(threshold):
        raise ValueError("分类阈值必须是开发队列确定的有限数值。")
    if set(labels) != {0, 1} or not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("指标需要两类结局，以及0～1之间的有效概率。")
    tn, fp, fn, tp = confusion_matrix(labels, probabilities >= threshold, labels=[0, 1]).ravel()
    return {"n": len(labels), "auc": float(roc_auc_score(labels, probabilities)), "threshold": float(threshold),
            "accuracy": float((tp + tn) / len(labels)), "sensitivity": float(tp / (tp + fn)),
            "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(labels, probabilities))}


## 提取特征和拟合模型
每个序列、每个标签逐层提取特征，再用四层的均值、标准差、最大值汇总到患者级。

| 特征 | 当前实现 |
| --- | --- |
| 强度分布、一阶统计、直方图 | MIRP statistics、intensity_histogram |
| 灰度共生矩阵、游程矩阵、区域大小矩阵 | MIRP GLCM、GLRLM、GLSZM |
| 灰度依赖、邻域灰度差 | MIRP NGLDM、NGTDM |
| 二维形状 | scikit-image：像素面积、Crofton周长、长短轴、等效直径、离心率、实心度、矩形占比 |

形状特征每个标签只保存一份。没有该标签时，相关特征留空，并保留标签像素数和出现层数比例。ROI不足二维范围时跳过形状和纹理，原因记在 `extraction_notes.csv`。同一标签的多个不相连区域整体作为这一层的 ROI。

归一化仍按单个序列的整张切片进行均值/标准差标准化，再乘100；固定灰度分箱宽度仍为25。GLCM和GLRLM使用二维方向特征平均，其他纹理使用二维矩阵。MIRP中的 `fixed_bin_size_pyradiomics` 是其内置的分箱算法名称，不会导入PyRadiomics；为了沿用这一分箱规则设置了 `ibsi_compliant=False`，因此本流程不声称整体满足IBSI标准。

换库后的特征名称、数量和部分定义会不同，尤其是二维形状及灰度依赖特征，不应当作原PyRadiomics结果的逐项复现。需要重新提取特征、重新训练影像组学模型；06仍使用相同的CSV/joblib路径。

训练两个模型：影像组学，以及影像组学＋五项临床信息。缺失值填补、标准化、特征筛选和正则化参数选择都放在训练折内部。开发队列五折预测用于选分类阈值；最终模型使用全部开发患者拟合。具体特征含义见[MIRP文档](https://oncoray.github.io/mirp/features_names.html)和[二维形状文档](https://scikit-image.org/docs/stable/api/skimage.measure.html#skimage.measure.regionprops)。


In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def shape_features_2d(mask):
    from skimage.measure import regionprops

    region = regionprops(mask.astype(np.uint8))[0]
    return {
        "area_pixels": float(region.area),
        "perimeter_pixels": float(region.perimeter_crofton),
        "major_axis_pixels": float(region.axis_major_length),
        "minor_axis_pixels": float(region.axis_minor_length),
        "equivalent_diameter_pixels": float(region.equivalent_diameter_area),
        "eccentricity": float(region.eccentricity),
        "solidity": float(region.solidity),
        "extent": float(region.extent),
    }


def extract_radiomics(cases, output, settings):
    from mirp import extract_features

    # MIRP的这个分箱选项沿用原来的固定宽度规则，不需要安装PyRadiomics。
    options = {
        "by_slice": True,
        "ibsi_compliant": False,
        "image_modality": "MR",
        "base_feature_families": ["statistics", "intensity_histogram", "glcm", "glrlm", "glszm", "ngldm", "ngtdm"],
        "base_discretisation_method": "fixed_bin_size_pyradiomics",
        "base_discretisation_bin_width": float(settings["binWidth"]),
        "intensity_normalisation": "standardisation" if settings["normalize"] else "none",
        "intensity_scaling": float(settings["normalizeScale"]) if settings["normalize"] else 1.0,
        "tissue_mask_type": "none",
        "bias_field_correction": False,
        "new_spacing": None,
        "anti_aliasing": False,
        "glcm_spatial_method": "2d_average",
        "glrlm_spatial_method": "2d_average",
        "glszm_spatial_method": "2d",
        "ngldm_spatial_method": "2d",
        "ngtdm_spatial_method": "2d",
        "write_features": False,
        "export_features": True,
        "parallel_backend": "none",
    }
    # 只取特征列，避免把MIRP返回的文件名、患者信息等元数据当作模型输入。
    prefixes = ("stat_", "ih_", "cm_", "rlm_", "szm_", "ngl_", "ngt_")
    output.mkdir(parents=True, exist_ok=True)
    folders, rows, notes = image_directories(), [], []
    for _, patient in cases.iterrows():
        images, labels = read_four_slices(patient, folders)
        row, values = {"mr": patient.mr}, {}
        for label in (1, 2, 3):
            row[f"roi{label}_pixels"] = int((labels == label).sum())
            row[f"roi{label}_slice_fraction"] = float(np.mean([(labels[:, :, s] == label).any() for s in range(4)]))
            for s in range(4):
                # nibabel数组为X,Y,Z；MIRP接收Z,Y,X，单层需保留长度为1的Z轴。
                mask = labels[:, :, s].T == label
                coordinates = np.argwhere(mask)
                if not len(coordinates):
                    continue
                if len(coordinates) < 2 or np.any(np.ptp(coordinates, axis=0) < 1):
                    notes.append({"mr": patient.mr, "slice": int(patient.max_layer) + s, "label": label,
                                  "reason": "ROI不足二维范围，形状和纹理留空"})
                    continue
                # 同一标注的形状只计算一次，不在三个序列中重复。
                for name, value in shape_features_2d(mask).items():
                    if np.isfinite(value):
                        values.setdefault(f"roi{label}_shape2d_{name}", []).append(value)
                for channel, sequence in enumerate(SEQUENCES):
                    try:
                        tables = extract_features(
                            image=np.ascontiguousarray(images[channel][:, :, s].T[None], dtype=np.float64),
                            mask=np.ascontiguousarray(mask[None], dtype=np.uint8),
                            roi_name="1", **options)
                        if not tables or tables[0] is None or len(tables[0]) != 1:
                            raise ValueError("MIRP未返回这一层ROI的单行特征。")
                        features = tables[0].iloc[0]
                        selected = [name for name in features.index if str(name).startswith(prefixes)]
                        if not selected:
                            raise ValueError("MIRP返回结果中没有找到强度或纹理特征。")
                        for name in selected:
                            value = float(features[name])
                            if np.isfinite(value):
                                values.setdefault(f"{sequence}_roi{label}_{name}", []).append(value)
                    except (RuntimeError, ValueError, TypeError) as exc:
                        notes.append({"mr": patient.mr, "slice": int(patient.max_layer) + s, "label": label,
                                      "reason": f"{sequence}: {exc}"})
                        pd.DataFrame(notes).to_csv(output / "extraction_notes.csv", index=False)
                        raise ValueError(f"{patient.mr}第{int(patient.max_layer) + s}层、标签{label}、{sequence}提取失败：{exc}") from exc
        for name, feature in values.items():
            row[name + "_mean"], row[name + "_std"], row[name + "_max"] = np.mean(feature), np.std(feature), np.max(feature)
        rows.append(row)
        print(f"影像组学 {len(rows)}/{len(cases)}：{patient.mr}", flush=True)
    pd.DataFrame(notes, columns=["mr", "slice", "label", "reason"]).to_csv(output / "extraction_notes.csv", index=False)
    table = pd.DataFrame(rows)
    table.to_csv(output / "features.csv", index=False)
    return table


def radiomics_search(rad_columns, clinical_columns):
    rad = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                    ("variance", VarianceThreshold()), ("scale", StandardScaler()), ("select", SelectPercentile(f_classif))])
    transforms = [("rad", rad, rad_columns)]
    if clinical_columns:
        transforms.append(("clinical", Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                                               ("scale", StandardScaler())]), clinical_columns))
    model = Pipeline([("features", ColumnTransformer(transforms)), ("lr", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(model, {"features__rad__select__percentile": [5, 10, 20], "lr__C": [.003, .01, .03, .1, .3, 1.]},
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring="roc_auc", n_jobs=1, error_score="raise")


def fit_radiomics(cases, output):
    features = pd.read_csv(output / "features.csv", dtype={"mr": str})
    if not features.mr.is_unique or set(features.mr) != set(cases.mr):
        raise ValueError("特征表与患者清单的MR编号不一致。")
    rad_columns = [column for column in features if column != "mr"]
    table = cases.merge(features, on="mr", validate="one_to_one").query("split == 'development'").reset_index(drop=True)
    y, folds = table.label.to_numpy(dtype=int), table.development_fold.to_numpy(dtype=int)
    models, summaries, predictions, selected_rows = {}, [], [], []
    for name, clinical in (("Radiomics", []), ("Radiomics_Clinical5", CLINICAL)):
        columns = rad_columns + clinical
        values = table[columns].replace([np.inf, -np.inf], np.nan)
        oof = np.zeros(len(table), dtype=float)
        for fold in range(1, 6):
            valid = folds == fold
            search = radiomics_search(rad_columns, clinical).fit(values.loc[~valid], y[~valid])
            oof[valid] = search.predict_proba(values.loc[valid])[:, 1]
            processor = search.best_estimator_.named_steps["features"].named_transformers_["rad"]
            retained = np.asarray(rad_columns)[processor.named_steps["variance"].get_support()]
            for feature in retained[processor.named_steps["select"].get_support()]:
                selected_rows.append({"model": name, "fold": fold, "feature": feature})
        threshold = choose_threshold(y, oof)
        models[name] = {"model": radiomics_search(rad_columns, clinical).fit(values, y).best_estimator_,
                        "threshold": threshold, "features": columns}
        frame = table[["mr", "label", "development_fold"]].copy()
        frame["model"], frame["probability"] = name, oof
        predictions.append(frame)
        summaries.append({"model": name, **classification_metrics(y, oof, threshold)})
    joblib.dump(models, output / "models.joblib")
    pd.concat(predictions).to_csv(output / "development_oof.csv", index=False)
    pd.DataFrame(summaries).to_csv(output / "development_metrics.csv", index=False)
    pd.DataFrame(selected_rows).to_csv(output / "selected_features.csv", index=False)
    return pd.DataFrame(summaries)


## 运行
这次换库后，将前面的 `RUN_EXTRACTION` 和 `RUN_TRAINING` 都设为 `True`，重新提取并训练。生成的 `features.csv` 和 `models.joblib` 要配套使用；已有分类四层缓存不需要重做。

完整提取会覆盖 `features.csv`；重新训练会覆盖 `models.joblib`。之后特征已提取好、只想重训时，可以只开启训练开关。


In [ ]:
if RUN_EXTRACTION:
    features = extract_radiomics(cases, OUTPUT, EXTRACTION)
    display(features.head())
if RUN_TRAINING:
    display(fit_radiomics(cases, OUTPUT))
if not RUN_EXTRACTION and not RUN_TRAINING:
    print("需要提取特征或训练时，修改前面的两个运行开关。")


## 传到算力机的文件
06 需要 `results/radiomics/features.csv` 和 `results/radiomics/models.joblib`。直接复制整个 `radiomics` 文件夹也可以，不需要压成专用数据包或登记版本。

算力机只读取特征表和模型做预测时不需要安装 MIRP 或 scikit-image；需要 pandas、numpy、scikit-learn 和 joblib。两台电脑的 scikit-learn 版本尽量一致，以便正常读取保存的模型。
